# Example vertebrae with landmarks, by life history strategy

Renders a 2 × N panel: rows are views, columns are life history strategies.
Rendering matches the PC warp grid (same renderers, material, rotation, crop).

In [ ]:
# Imports and paths

import os, re, json, gc, ast
import numpy as np
import pandas as pd
import cv2
import pyvista as pv
import open3d as o3d
from collections import defaultdict
from pathlib import Path
from NSM.plotting import load_mrk_json
from NSM.helper_funcs import pv_to_o3d, render_cameras

# Specify training directory and atlas directory
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"  # atlas/builder run that produced alignedLMs
DROPBOX_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")

# Build other directories relative to those above
cwd      = Path.cwd()
base_wd  = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

LM_DIR = DROPBOX_ROOT / ATLAS_RUN / "alignedLMs"

OUT_DIR = Path("life_hist_exemplars")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")


# ── Which specimen represents each trait ─────────────────────────────────────
EXEMPLARS = {
    "arboreal": ("chamaeleonidae_chamaeleo_calyptratus_uf-191369", "c5"),
    "burrowing": ("amphisbaenidae_bipes_biporus_uf42060", "l5"),
    "grass-swimmer": ("gerrhosauridae_tetradactylus_tetradactylus_ummz", "c6"),
    "saxicolous": ("agamidae_agama_atra_uf180711", "c5"), 
    "terrestrial": ("hoplocercidae_enyaloides_oshaughnessyi_uf191439", "c5"),
    "snake": ("homalopsidae_homalopsis_buccata_uf61845", "t10")}

# ── Config — matches the warp grid script ────────────────────────────────────
WIDTH, HEIGHT = 640, 480
ROT_MATRIX    = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(13)])

LM_RADIUS_FRAC = 0.022      # sphere radius as a fraction of mesh bounding-box diagonal
LM_COLOR       = [0.85, 0.16, 0.16]     # landmark spheres
BONE_COLOR     = [0.82, 0.82, 0.82]     # vertebra surface
OUT_PANEL = OUT_DIR / "exemplar_vertebrae_landmarks.png"

RENDERERS = [o3d.visualization.rendering.OffscreenRenderer(WIDTH, HEIGHT)
             for _ in range(4)]

# Load config and parse species / vertebra from filenames (same logic as PCA_tSNE_UMAP.ipynb)
config_path = "model_params_config.json"
with open(config_path) as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from {config_path}\033[0m")

# Parse filenames
all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]
print(f"{len(all_vtk_files)} meshes listed in config")

pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)

In [ ]:
# ── Crops: render_cameras returns a 2×2 grid of camera views ─────────────────
def _crop(combined, which):
    if which == "top_left":     return combined[:HEIGHT, :WIDTH]
    if which == "top_right":    return combined[:HEIGHT, WIDTH:]
    if which == "bottom_left":  return combined[HEIGHT:, :WIDTH]
    if which == "bottom_right": return combined[HEIGHT:, WIDTH:]
    raise ValueError(which)


# Each view is (label, crop, extra z-rotation in degrees applied before rendering).
VIEWS = [
    ("SIDE", "top_right",   0),
    ("BACK", "bottom_left", 180),
]


# ── Locate mesh + landmark files for a (specimen_id, vertebra) pair ──────────
def _find_files(specimen_id, vertebra):
    """
    Returns (mesh_path, lm_path).

    The mesh filename is read out of `specimens` rather than rebuilt from its
    parts -- specimen IDs can contain hyphens (uf-191369), so specimen_id and
    vertebra do not concatenate back into the original name.
    """
    match = specimens[
        (specimens["specimen_id"].str.lower() == str(specimen_id).lower())
        & (specimens["vertebra"].str.lower() == str(vertebra).lower())
    ]
    if match.empty:
        raise KeyError(
            f"No row in `specimens` for ({specimen_id!r}, {vertebra!r}). "
            f"Use a pair printed by the browse cell."
        )
    if len(match) > 1:
        print(f"    note: ({specimen_id}, {vertebra}) matched {len(match)} rows, "
              f"using {match.iloc[0]['mesh']}")

    mesh_name = match.iloc[0]["mesh"]
    stem_want = Path(mesh_name).stem.lower()

    hits = [p for p in cfg["list_mesh_paths"]
            if Path(p).stem.lower() == stem_want]
    if not hits:                       # extensions may differ between the two
        hits = [p for p in cfg["list_mesh_paths"]
                if stem_want in Path(p).stem.lower()]
    if not hits:
        raise FileNotFoundError(
            f"`specimens` lists '{mesh_name}' but no matching path is in "
            f"cfg['list_mesh_paths']."
        )

    mesh_path = Path(hits[0])
    return mesh_path, LM_DIR / (mesh_path.stem + ".mrk.json")


def _make_material():
    mat = o3d.visualization.rendering.MaterialRecord()
    mat.shader     = "defaultLit"
    mat.base_color = [1.0, 1.0, 1.0, 1.0]      # white — vertex colours carry the tint
    return mat


# Meshes render on key green; the vignette replaces those pixels afterwards.
def _bg_for_trait(trait):
    return [0.0, 1.0, 0.0]

VIG_INNER = 0.18     # fraction of cell half-diagonal that stays pure white
VIG_OUTER = 0.95     # fraction at which the trait colour reaches full opacity
KEY_TOL   = 60       # masking tolerance for the key colour

def _radial_ramp(h, w, inner=VIG_INNER, outer=VIG_OUTER):
    """0 at the centre, 1 at the edges, smoothstepped between inner and outer."""
    yy, xx = np.mgrid[0:h, 0:w]
    cy, cx = (h - 1) / 2, (w - 1) / 2
    d      = np.sqrt(((yy - cy) / cy) ** 2 + ((xx - cx) / cx) ** 2) / np.sqrt(2)
    t      = np.clip((d - inner) / (outer - inner), 0, 1)
    return t * t * (3 - 2 * t)           # smoothstep, avoids a hard ring

def _vignette(cell, trait, ramp=None, tol=KEY_TOL):
    """Replace background with white-centre → trait-colour vignette.
    Background colour is sampled from the cell corner, so it doesn't matter
    what Open3D's tone mapping did to the nominal key colour."""
    h, w = cell.shape[:2]
    if ramp is None:
        ramp = _radial_ramp(h, w)

    key = cell[2, 2].astype(int)          # corner is always background
    col_bgr = np.array(trait_colors.get(trait, (0.5, 0.5, 0.5))[::-1]) * 255
    grad = (255 * (1 - ramp[..., None]) + col_bgr * ramp[..., None]).astype(np.uint8)

    mask = (np.abs(cell.astype(int) - key) < tol).all(axis=2)
    out = cell.copy()
    out[mask] = grad[mask]
    return out

def _z_rotate(mesh, degrees):
    """Spin a mesh about the vertical axis, in place, about its own centre."""
    if degrees:
        R = o3d.geometry.get_rotation_matrix_from_axis_angle(
            [0, 0, np.deg2rad(degrees)])
        mesh.rotate(R, center=mesh.get_center())
    return mesh

def _build_mesh_with_landmarks(mesh_path, lm_path):
    """Vertebra + landmark spheres merged into one Open3D mesh with vertex colours."""
    pv_mesh = pv.read(str(mesh_path))
    pv_mesh = pv_mesh.extract_surface(algorithm="dataset_surface").triangulate()
    pv_mesh = pv_mesh.compute_normals(cell_normals=False, point_normals=True,
                                      inplace=False, auto_orient_normals=True)
    bone = pv_to_o3d(pv_mesh)
    bone.compute_vertex_normals()
    bone.paint_uniform_color(BONE_COLOR)

    # Sphere size scales with the specimen so all columns look consistent
    bbox   = bone.get_axis_aligned_bounding_box()
    diag   = float(np.linalg.norm(bbox.get_extent()))
    radius = diag * LM_RADIUS_FRAC

    coords, _ = load_mrk_json(lm_path)
    combined  = bone
    for pt in np.asarray(coords):
        sph = o3d.geometry.TriangleMesh.create_sphere(radius=radius, resolution=12)
        sph.translate(pt)
        sph.compute_vertex_normals()
        sph.paint_uniform_color(LM_COLOR)
        combined += sph

    combined.rotate(ROT_MATRIX, center=combined.get_center())
    return combined, len(coords)

# Build specimen metadata table -- species, family, life history, vertebral region

SPECIES_CSV = "../lizard_species_list.csv"

# Parse specimen ID and vertebra from filenames
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

# Join against the species master list
sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'\"")
sdf["color"]  = sdf["color"].apply(ast.literal_eval)
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")

# Region from vertebra letter code
REGION_NAMES = {"C": "CERVICAL", "T": "THORACIC", "L": "LUMBAR"}
specimens["region"] = specimens["vertebra"].str[0].map(REGION_NAMES)

print("Specimens dataframe head:\n", specimens.head())

# One color per trait, derived from that trait's marker 
markers = ['P', '+', 's', 'd', 'X', 'o', '2']
colors = [(0.65, 0.69, 0.12),   # pea soup
        (0.84, 0.65, 0.23),   # saffron
        (0.72, 0.44, 0.22),   # mud
        (0.36, 0.557, 0.68),  # powder blue
        (0.10, 0.51, 0.40),   # deep blue
        (0.60, 0.50, 0.46),   # slate
        (0, 0, 0)]            # black
marker_to_color = dict(zip(markers, colors))

# Match against traits in species_lis.csv
trait_marker = specimens.drop_duplicates("trait").set_index("trait")["marker"]
trait_colors = {t: marker_to_color.get(m, (0.5, 0.5, 0.5))
                for t, m in trait_marker.items() if pd.notna(t)}
specimens["trait_color"] = specimens["trait"].map(trait_colors)
unmapped = [t for t, m in trait_marker.items() if pd.notna(t) and m not in marker_to_color]
if unmapped:
    print(f"\033[33mTraits using a marker not in marker_to_color (defaulting to grey): {unmapped}\033[0m")
print(f"{len(trait_colors)} traits: {sorted(trait_colors)}")

## Render the panel

In [ ]:
# ── Render every exemplar, both views ────────────────────────────────────────

mat = _make_material()

missing = [t for t, (s, v) in EXEMPLARS.items() if "TO_DO" in (s, v)]
if missing:
    raise ValueError(f"EXEMPLARS still unset for: {missing}. Run the browse cell.")

by_rotation = defaultdict(list)
for view_lbl, crop_key, deg in VIEWS:
    by_rotation[deg].append((view_lbl, crop_key))

cells = {}
for trait, (specimen_id, vertebra) in EXEMPLARS.items():
    bg    = _bg_for_trait(trait)
    blank = np.full((HEIGHT, WIDTH, 3),
                    (np.array(bg) * 255).astype(np.uint8), dtype=np.uint8)

    for r in RENDERERS:                      # background follows the trait
        r.scene.set_background(list(bg) + [1.0])

    try:
        mesh_path, lm_path = _find_files(specimen_id, vertebra)

        for deg, views in by_rotation.items():
            o3d_mesh, n_lms = _build_mesh_with_landmarks(mesh_path, lm_path)
            _z_rotate(o3d_mesh, deg)
            panels = render_cameras(RENDERERS, o3d_mesh, 0, mat, 1, n_rotations=1)

            for view_lbl, crop_key in views:
                cells[(view_lbl, trait)] = _crop(panels, crop_key).copy()

            del o3d_mesh, panels
            gc.collect()

        print(f"  {trait:<15s} {specimen_id}-{vertebra}  ({n_lms} landmarks)  ✓",
              flush=True)

    except Exception as e:
        print(f"  {trait:<15s} FAILED: {e}")
        for view_lbl, _, _ in VIEWS:
            cells[(view_lbl, trait)] = blank
    gc.collect()


# ── Assemble 4 rows × 3 columns ──────────────────────────────────────────────
# Traits 1-3 fill rows 0-1 (SIDE then BACK), traits 4-6 fill rows 2-3.
traits  = list(EXEMPLARS)
N_COLS  = 3
blocks  = [traits[i:i + N_COLS] for i in range(0, len(traits), N_COLS)]

RAMP = _radial_ramp(HEIGHT, WIDTH)       # same for every cell, compute once

rows = []
for block in blocks:
    for view_lbl, _, _ in VIEWS:
        rows.append(np.hstack([
            _vignette(cells[(view_lbl, t)], t, RAMP) for t in block
        ]))
panel = np.vstack(rows)

# ── Labels drawn onto the panel ──────────────────────────────────────────────
FONT        = cv2.FONT_HERSHEY_DUPLEX
TRAIT_SCALE = 1.1        # trait name, top-left of each SIDE cell
VIEW_SCALE  = 1.1        # SIDE / BACK, bottom-left of every cell
THICK       = 2
PAD         = 18         # inset from the cell edge

def _text_color(bgr_cell):
    """Dark text on light backgrounds, light text on dark ones."""
    lum = bgr_cell[..., 0].mean() * 0.114 + \
          bgr_cell[..., 1].mean() * 0.587 + \
          bgr_cell[..., 2].mean() * 0.299
    return (40, 40, 40) if lum > 130 else (235, 235, 235)

for b, block in enumerate(blocks):
    for v, (view_lbl, _, _) in enumerate(VIEWS):
        row = b * len(VIEWS) + v
        y0  = row * HEIGHT
        for j, trait in enumerate(block):
            x0   = j * WIDTH
            cell = panel[y0:y0 + HEIGHT, x0:x0 + WIDTH]
            col  = _text_color(cell)

            # Trait name once per pair, on the SIDE row
            if v == 0:
                cv2.putText(panel, trait.upper(),
                            (x0 + PAD, y0 + PAD + 34),
                            FONT, TRAIT_SCALE, col, THICK, cv2.LINE_AA)

            # View label on every cell
            cv2.putText(panel, view_lbl,
                        (x0 + PAD, y0 + HEIGHT - PAD),
                        FONT, VIEW_SCALE, col, THICK, cv2.LINE_AA)

cv2.imwrite(str(OUT_PANEL), panel)
print(f"\nSaved → {OUT_PANEL}   ({panel.shape[1]}×{panel.shape[0]} px)")